In [ ]:
## ULTIMATE Comprehensive Dataset Builder
## Includes EVERYTHING: Floor Pivots, Overnight touches, Initial Balance, ALL features

import pandas as pd
import numpy as np
from datetime import datetime

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

# Adjust the index to group by NY trading day
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

print("="*80)
print("ULTIMATE COMPREHENSIVE DATASET BUILDER")
print("="*80)
print(f"Analyzing {len(dates)-1} trading days from {start_date.date()} to {end_date}")
print("\nFeatures being captured:")
print("  ✓ Floor Pivots (PP, R1-R3, S1-S3, Golden Zones)")
print("  ✓ Opening position relative to floor pivots")
print("  ✓ Overnight pivot touches (which levels hit before 9:30)")
print("  ✓ Previous day/week high/low touches during overnight")
print("  ✓ All hourly ranges + 4H candles (pre-market only)")
print("  ✓ All session patterns + ORBs")
print("  ✓ Initial Balance (9:30-10:30am)")
print("  ✓ NY ORB (9:30-9:45am) + 9:35 ORB (9:30-9:35am)")
print("  ✓ Level touches before 10:30am (prev day/week high/low)")
print("  ✓ Gap analysis vs prev close")
print("  ✓ 10am-2pm candle direction (outcome for 10am entry predictions)")
print("  ✓ 10am-2pm first extreme (high or low formed first)")
print("  NOTE: 10am-2pm and 2pm-5pm candles excluded as FEATURES (too late for prediction)")
print()

# ============================================================
# FLOOR PIVOT FUNCTIONS
# ============================================================

def calculate_floor_pivots(high, low, close):
    """Calculate floor pivot levels"""
    pp = (high + low + close) / 3
    
    r1 = (2 * pp) - low
    r2 = pp + (high - low)
    r3 = high + 2 * (pp - low)
    
    s1 = (2 * pp) - high
    s2 = pp - (high - low)
    s3 = low - 2 * (high - pp)
    
    # Golden Zone (0.5-0.618 of previous day range)
    pivot_range = high - low
    gz_low = low + pivot_range * 0.5
    gz_high = low + pivot_range * 0.618
    
    return {
        'PP': pp, 'R1': r1, 'R2': r2, 'R3': r3,
        'S1': s1, 'S2': s2, 'S3': s3,
        'GZ_LOW': gz_low, 'GZ_HIGH': gz_high
    }

def get_pivot_position(price, pivots, threshold_pct=0.1):
    """Determine position relative to floor pivots"""
    threshold = pivots['PP'] * (threshold_pct / 100)
    
    if abs(price - pivots['PP']) <= threshold:
        return 'at_pp'
    elif pivots['GZ_LOW'] <= price <= pivots['GZ_HIGH']:
        return 'in_gz'
    elif price >= pivots['R3']:
        return 'above_r3'
    elif price >= pivots['R2']:
        return 'r2_to_r3'
    elif price >= pivots['R1']:
        return 'r1_to_r2'
    elif price > pivots['GZ_HIGH']:
        return 'gz_to_r1'
    elif price > pivots['PP']:
        return 'pp_to_gz'
    elif price <= pivots['S3']:
        return 'below_s3'
    elif price <= pivots['S2']:
        return 's3_to_s2'
    elif price <= pivots['S1']:
        return 's2_to_s1'
    elif price < pivots['GZ_LOW']:
        return 's1_to_gz'
    else:
        return 'gz_to_pp'

def check_level_touch(data, level_price, threshold_pct=0.05):
    """Check if a price level was touched in given data"""
    if data.empty:
        return False
    threshold = level_price * (threshold_pct / 100)
    for _, row in data.iterrows():
        if row['Low'] - threshold <= level_price <= row['High'] + threshold:
            return True
    return False

# ============================================================
# BUILD COMPREHENSIVE DATASET
# ============================================================

unified_data = []
skipped_count = 0

for idx, date in enumerate(dates[1:], 1):
    if idx % 500 == 0:
        print(f"  Processing day {idx}...")
    
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define trading day boundaries
    ny_trading_day_start = actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    ny_trading_day_end = actual_date.replace(hour=17, minute=0, second=0)
    
    full_day_data = df.loc[ny_trading_day_start:ny_trading_day_end]
    if full_day_data.empty:
        skipped_count += 1
        continue
    
    # ========================================
    # PREVIOUS DAY DATA
    # ========================================
    prev_date = date - pd.Timedelta(days=1)
    prev_actual_date = prev_date + pd.Timedelta(hours=7)
    prev_day_start = prev_actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    prev_day_end = prev_actual_date.replace(hour=17, minute=0, second=0)
    prev_day_data = df.loc[prev_day_start:prev_day_end]
    
    if not prev_day_data.empty:
        prev_day_high = prev_day_data['High'].max()
        prev_day_low = prev_day_data['Low'].min()
        prev_day_open = prev_day_data['Open'].iloc[0]
        prev_day_close = prev_day_data['Close'].iloc[-1]
        prev_day_direction = 'bullish' if prev_day_close > prev_day_open else 'bearish'
    else:
        prev_day_high = prev_day_low = prev_day_open = prev_day_close = prev_day_direction = None
    
    # ========================================
    # PREVIOUS WEEK DATA
    # ========================================
    current_day_index = list(dates).index(date)
    if current_day_index >= 5:
        prev_week_dates = dates[current_day_index-5:current_day_index]
        prev_week_data_list = []
        
        for pwd in prev_week_dates:
            pwd_actual = pwd + pd.Timedelta(hours=7)
            pwd_start = pwd_actual.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
            pwd_end = pwd_actual.replace(hour=17, minute=0, second=0)
            pwd_data = df.loc[pwd_start:pwd_end]
            if not pwd_data.empty:
                prev_week_data_list.append(pwd_data)
        
        if prev_week_data_list:
            prev_week_combined = pd.concat(prev_week_data_list)
            prev_week_high = prev_week_combined['High'].max()
            prev_week_low = prev_week_combined['Low'].min()
        else:
            prev_week_high = prev_week_low = None
    else:
        prev_week_high = prev_week_low = None
    
    # ========================================
    # FLOOR PIVOTS (from previous day)
    # ========================================
    if prev_day_high and prev_day_low and prev_day_close:
        floor_pivots = calculate_floor_pivots(prev_day_high, prev_day_low, prev_day_close)
    else:
        floor_pivots = None
    
    # ========================================
    # OVERNIGHT SESSION (6pm-9:30am)
    # ========================================
    overnight_start = ny_trading_day_start
    overnight_end = actual_date.replace(hour=9, minute=30, second=0)
    overnight_data = df.loc[overnight_start:overnight_end]
    
    if not overnight_data.empty:
        overnight_high = overnight_data['High'].max()
        overnight_low = overnight_data['Low'].min()
        overnight_open = overnight_data['Open'].iloc[0]
        overnight_close = overnight_data['Close'].iloc[-1]
        overnight_direction = 'bullish' if overnight_close > overnight_open else 'bearish'
        
        # Check if prev day/week levels touched overnight
        overnight_touched_prev_day_high = check_level_touch(overnight_data, prev_day_high) if prev_day_high else False
        overnight_touched_prev_day_low = check_level_touch(overnight_data, prev_day_low) if prev_day_low else False
        overnight_touched_prev_week_high = check_level_touch(overnight_data, prev_week_high) if prev_week_high else False
        overnight_touched_prev_week_low = check_level_touch(overnight_data, prev_week_low) if prev_week_low else False
        
        # Check which floor pivots touched overnight
        if floor_pivots:
            overnight_touched_r3 = check_level_touch(overnight_data, floor_pivots['R3'])
            overnight_touched_r2 = check_level_touch(overnight_data, floor_pivots['R2'])
            overnight_touched_r1 = check_level_touch(overnight_data, floor_pivots['R1'])
            overnight_touched_pp = check_level_touch(overnight_data, floor_pivots['PP'])
            overnight_touched_s1 = check_level_touch(overnight_data, floor_pivots['S1'])
            overnight_touched_s2 = check_level_touch(overnight_data, floor_pivots['S2'])
            overnight_touched_s3 = check_level_touch(overnight_data, floor_pivots['S3'])
            overnight_touched_gz = (overnight_data['Low'].min() <= floor_pivots['GZ_HIGH'] and 
                                   overnight_data['High'].max() >= floor_pivots['GZ_LOW'])
            
            # Highest/lowest pivot touched overnight
            overnight_highest_pivot = None
            overnight_lowest_pivot = None
            for level in ['R3', 'R2', 'R1']:
                if check_level_touch(overnight_data, floor_pivots[level]):
                    overnight_highest_pivot = level
                    break
            for level in ['S3', 'S2', 'S1']:
                if check_level_touch(overnight_data, floor_pivots[level]):
                    overnight_lowest_pivot = level
                    break
        else:
            overnight_touched_r3 = overnight_touched_r2 = overnight_touched_r1 = False
            overnight_touched_pp = overnight_touched_s1 = overnight_touched_s2 = overnight_touched_s3 = False
            overnight_touched_gz = False
            overnight_highest_pivot = overnight_lowest_pivot = None
    else:
        overnight_high = overnight_low = overnight_open = overnight_direction = None
        overnight_touched_prev_day_high = overnight_touched_prev_day_low = False
        overnight_touched_prev_week_high = overnight_touched_prev_week_low = False
        overnight_touched_r3 = overnight_touched_r2 = overnight_touched_r1 = False
        overnight_touched_pp = overnight_touched_s1 = overnight_touched_s2 = overnight_touched_s3 = False
        overnight_touched_gz = False
        overnight_highest_pivot = overnight_lowest_pivot = None
    
    # ========================================
    # ASIA SESSION
    # ========================================
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    asian_data = df.loc[asian_start:asian_end]
    
    if not asian_data.empty:
        asia_high = asian_data['High'].max()
        asia_low = asian_data['Low'].min()
        asia_open = asian_data['Open'].iloc[0]
        asia_close = asian_data['Close'].iloc[-1]
        asia_direction = 'bullish' if asia_close > asia_open else 'bearish'
    else:
        asia_high = asia_low = asia_open = asia_close = asia_direction = None
    
    # ========================================
    # LONDON SESSION
    # ========================================
    london_start = actual_date.replace(hour=2, minute=0, second=0)
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    london_data = df.loc[london_start:london_end]
    
    if not london_data.empty:
        london_high = london_data['High'].max()
        london_low = london_data['Low'].min()
        london_open = london_data['Open'].iloc[0]
        london_close = london_data['Close'].iloc[-1]
        london_direction = 'bullish' if london_close > london_open else 'bearish'
    else:
        london_high = london_low = london_open = london_close = london_direction = None
    
    # ========================================
    # SESSION PATTERN
    # ========================================
    if asia_high and london_high:
        if london_high > asia_high and london_low < asia_low:
            session_pattern = 'london_engulfs_asia'
        elif london_high <= asia_high and london_low >= asia_low:
            session_pattern = 'asia_engulfs_london'
        elif london_high > asia_high and london_low >= asia_low:
            session_pattern = 'london_partial_up'
        elif london_low < asia_low and london_high <= asia_high:
            session_pattern = 'london_partial_down'
        else:
            session_pattern = 'other'
    else:
        session_pattern = None
    
    # ========================================
    # HOURLY RANGES
    # ========================================
    hourly_ranges = {}
    
    # Globex 6pm
    globex_start = actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    globex_end = globex_start + pd.Timedelta(hours=1)
    globex_data = df.loc[globex_start:globex_end]
    if not globex_data.empty:
        hourly_ranges['globex_6pm'] = {
            'direction': 'bullish' if globex_data['Close'].iloc[-1] > globex_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['globex_6pm'] = None
    
    # Asia 8pm
    asia_hour_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asia_hour_end = asia_hour_start + pd.Timedelta(hours=1)
    asia_hour_data = df.loc[asia_hour_start:asia_hour_end]
    if not asia_hour_data.empty:
        hourly_ranges['asia_8pm'] = {
            'direction': 'bullish' if asia_hour_data['Close'].iloc[-1] > asia_hour_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['asia_8pm'] = None
    
    # Midnight
    midnight_start = actual_date.replace(hour=0, minute=0, second=0)
    midnight_end = midnight_start + pd.Timedelta(hours=1)
    midnight_data = df.loc[midnight_start:midnight_end]
    if not midnight_data.empty:
        hourly_ranges['midnight'] = {
            'direction': 'bullish' if midnight_data['Close'].iloc[-1] > midnight_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['midnight'] = None
    
    # London 2am
    london_hour_start = actual_date.replace(hour=2, minute=0, second=0)
    london_hour_end = london_hour_start + pd.Timedelta(hours=1)
    london_hour_data = df.loc[london_hour_start:london_hour_end]
    if not london_hour_data.empty:
        hourly_ranges['london_2am'] = {
            'direction': 'bullish' if london_hour_data['Close'].iloc[-1] > london_hour_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['london_2am'] = None
    
    # 7am
    hour_7am_start = actual_date.replace(hour=7, minute=0, second=0)
    hour_7am_end = hour_7am_start + pd.Timedelta(hours=1)
    hour_7am_data = df.loc[hour_7am_start:hour_7am_end]
    if not hour_7am_data.empty:
        hourly_ranges['hour_7am'] = {
            'direction': 'bullish' if hour_7am_data['Close'].iloc[-1] > hour_7am_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['hour_7am'] = None
    
    # 8am
    hour_8am_start = actual_date.replace(hour=8, minute=0, second=0)
    hour_8am_end = hour_8am_start + pd.Timedelta(hours=1)
    hour_8am_data = df.loc[hour_8am_start:hour_8am_end]
    if not hour_8am_data.empty:
        hourly_ranges['ny_futures_8am'] = {
            'direction': 'bullish' if hour_8am_data['Close'].iloc[-1] > hour_8am_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['ny_futures_8am'] = None
    
    # ========================================
    # 4H CANDLES (Only those complete before useful entry times)
    # ========================================
    four_hour_candles = {}
    
    candle_configs = [
        ('6pm', 18, 22, -1),   # 6pm-10pm previous day - complete by 10pm
        ('10pm', 22, 26, -1),  # 10pm-2am - complete by 2am
        ('2am', 2, 6, 0),      # 2am-6am - complete by 6am
        ('6am', 6, 10, 0),     # 6am-10am - complete by 10am (for 10:30+ entries only)
        # REMOVED: ('10am', 10, 14, 0) - completes at 2pm, too late for prediction
        # REMOVED: ('2pm', 14, 17, 0) - completes at 5pm, way too late
    ]
    
    for candle_name, start_hour, end_hour, day_offset in candle_configs:
        base_date = actual_date + pd.Timedelta(days=day_offset)
        candle_start = base_date.replace(hour=start_hour, minute=0, second=0)
        
        if end_hour >= 24:
            candle_end = (base_date + pd.Timedelta(days=1)).replace(hour=end_hour-24, minute=0, second=0)
        else:
            candle_end = base_date.replace(hour=end_hour, minute=0, second=0)
        
        candle_data = df.loc[candle_start:candle_end]
        
        if not candle_data.empty:
            c_open = candle_data['Open'].iloc[0]
            c_close = candle_data['Close'].iloc[-1]
            four_hour_candles[f'candle_4h_{candle_name}'] = {
                'direction': 'bullish' if c_close > c_open else 'bearish'
            }
        else:
            four_hour_candles[f'candle_4h_{candle_name}'] = None
    
    # ========================================
    # 6-9AM RANGE
    # ========================================
    range_6_9_start = actual_date.replace(hour=6, minute=0, second=0)
    range_6_9_end = actual_date.replace(hour=9, minute=0, second=0)
    range_6_9_data = df.loc[range_6_9_start:range_6_9_end]
    
    if not range_6_9_data.empty:
        r69_open = range_6_9_data['Open'].iloc[0]
        r69_close = range_6_9_data['Close'].iloc[-1]
        range_6_9_direction = 'bullish' if r69_close > r69_open else 'bearish'
    else:
        range_6_9_direction = None
    
    # ========================================
    # PRE-MARKET ORBS (before 9:30)
    # ========================================
    orb_periods = {}
    
    # Globex ORB (6pm+15)
    if hourly_ranges['globex_6pm']:
        orb_globex_start = globex_start
        orb_globex_end = orb_globex_start + pd.Timedelta(minutes=15)
        orb_globex_data = df.loc[orb_globex_start:orb_globex_end]
        if len(orb_globex_data) >= 10:
            orb_periods['globex_orb'] = {
                'direction': 'bullish' if orb_globex_data['Close'].iloc[-1] > orb_globex_data['Open'].iloc[0] else 'bearish'
            }
        else:
            orb_periods['globex_orb'] = None
    else:
        orb_periods['globex_orb'] = None
    
    # Asia ORB (8pm+15)
    if hourly_ranges['asia_8pm']:
        orb_asia_start = asia_hour_start
        orb_asia_end = orb_asia_start + pd.Timedelta(minutes=15)
        orb_asia_data = df.loc[orb_asia_start:orb_asia_end]
        if len(orb_asia_data) >= 10:
            orb_periods['asia_orb'] = {
                'direction': 'bullish' if orb_asia_data['Close'].iloc[-1] > orb_asia_data['Open'].iloc[0] else 'bearish'
            }
        else:
            orb_periods['asia_orb'] = None
    else:
        orb_periods['asia_orb'] = None
    
    # London ORB (2am+15)
    if hourly_ranges['london_2am']:
        orb_london_start = london_hour_start
        orb_london_end = orb_london_start + pd.Timedelta(minutes=15)
        orb_london_data = df.loc[orb_london_start:orb_london_end]
        if len(orb_london_data) >= 10:
            orb_periods['london_orb'] = {
                'direction': 'bullish' if orb_london_data['Close'].iloc[-1] > orb_london_data['Open'].iloc[0] else 'bearish'
            }
        else:
            orb_periods['london_orb'] = None
    else:
        orb_periods['london_orb'] = None
    
    # PreMarket ORB (8am+15)
    if hourly_ranges['ny_futures_8am']:
        orb_premarket_start = hour_8am_start
        orb_premarket_end = orb_premarket_start + pd.Timedelta(minutes=15)
        orb_premarket_data = df.loc[orb_premarket_start:orb_premarket_end]
        if len(orb_premarket_data) >= 10:
            orb_periods['premarket_orb'] = {
                'direction': 'bullish' if orb_premarket_data['Close'].iloc[-1] > orb_premarket_data['Open'].iloc[0] else 'bearish'
            }
        else:
            orb_periods['premarket_orb'] = None
    else:
        orb_periods['premarket_orb'] = None
    
    # ========================================
    # NY CASH SESSION (9:30-4pm)
    # ========================================
    ny_cash_start = actual_date.replace(hour=9, minute=30, second=0)
    ny_cash_end = actual_date.replace(hour=16, minute=0, second=0)
    ny_cash_data = df.loc[ny_cash_start:ny_cash_end]
    
    if not ny_cash_data.empty:
        ny_open = ny_cash_data['Open'].iloc[0]
        ny_close = ny_cash_data['Close'].iloc[-1]
        ny_high = ny_cash_data['High'].max()
        ny_low = ny_cash_data['Low'].min()
        ny_direction = 'bullish' if ny_close > ny_open else 'bearish'
        ny_range_pct = ((ny_high - ny_low) / ny_low) * 100
    else:
        ny_open = ny_close = ny_high = ny_low = ny_direction = ny_range_pct = None
    
    # ========================================
    # NY ORB (9:30-9:45) - for POST-ORB predictions
    # ========================================
    ny_orb_start = actual_date.replace(hour=9, minute=30, second=0)
    ny_orb_end = actual_date.replace(hour=9, minute=45, second=0)
    ny_orb_data = df.loc[ny_orb_start:ny_orb_end]
    
    if len(ny_orb_data) >= 10:
        orb_high = ny_orb_data['High'].max()
        orb_low = ny_orb_data['Low'].min()
        orb_open = ny_orb_data['Open'].iloc[0]
        orb_close = ny_orb_data['Close'].iloc[-1]
        ny_orb_direction = 'bullish' if orb_close > orb_open else 'bearish'
        
        # Check which side breaks first
        rest_of_day_start = ny_orb_end
        rest_of_day_end = actual_date.replace(hour=16, minute=0, second=0)
        rest_of_day = df.loc[rest_of_day_start:rest_of_day_end]
        
        orb_first_break = None
        if not rest_of_day.empty:
            for idx, row in rest_of_day.iterrows():
                if orb_first_break is None:
                    if row['High'] > orb_high:
                        orb_first_break = 'high'
                    elif row['Low'] < orb_low:
                        orb_first_break = 'low'
                if orb_first_break:
                    break
    else:
        ny_orb_direction = orb_first_break = None
    
    # ========================================
    # 9:30-9:35 ORB (5-minute) - for 9:45+ predictions
    # ========================================
    orb_935_start = actual_date.replace(hour=9, minute=30, second=0)
    orb_935_end = actual_date.replace(hour=9, minute=35, second=0)
    orb_935_data = df.loc[orb_935_start:orb_935_end]
    
    if len(orb_935_data) >= 3:
        orb935_high = orb_935_data['High'].max()
        orb935_low = orb_935_data['Low'].min()
        orb935_open = orb_935_data['Open'].iloc[0]
        orb935_close = orb_935_data['Close'].iloc[-1]
        orb_935_direction = 'bullish' if orb935_close > orb935_open else 'bearish'
        
        # Check which side breaks first after 9:35
        rest_after_935_start = orb_935_end
        rest_after_935_end = actual_date.replace(hour=16, minute=0, second=0)
        rest_after_935 = df.loc[rest_after_935_start:rest_after_935_end]
        
        orb_935_first_break = None
        if not rest_after_935.empty:
            for idx, row in rest_after_935.iterrows():
                if orb_935_first_break is None:
                    if row['High'] > orb935_high:
                        orb_935_first_break = 'high'
                    elif row['Low'] < orb935_low:
                        orb_935_first_break = 'low'
                if orb_935_first_break:
                    break
    else:
        orb_935_direction = orb_935_first_break = None
    
    # ========================================
    # INITIAL BALANCE (9:30-10:30) - for POST-IB predictions
    # ========================================
    ib_start = actual_date.replace(hour=9, minute=30, second=0)
    ib_end = actual_date.replace(hour=10, minute=30, second=0)
    ib_data = df.loc[ib_start:ib_end]
    
    if not ib_data.empty:
        ib_high = ib_data['High'].max()
        ib_low = ib_data['Low'].min()
        ib_open = ib_data['Open'].iloc[0]
        ib_close = ib_data['Close'].iloc[-1]
        ib_direction = 'bullish' if ib_close > ib_open else 'bearish'
        ib_range_pct = ((ib_high - ib_low) / ib_low) * 100
    else:
        ib_high = ib_low = ib_open = ib_direction = ib_range_pct = None
    
    # ========================================
    # 10AM-2PM CANDLE (for 10am entry predictions)
    # This is an OUTCOME variable - predicting this candle's direction
    # ========================================
    candle_10am_start = actual_date.replace(hour=10, minute=0, second=0)
    candle_10am_end = actual_date.replace(hour=14, minute=0, second=0)
    candle_10am_data = df.loc[candle_10am_start:candle_10am_end]
    
    if not candle_10am_data.empty:
        candle_10am_open = candle_10am_data['Open'].iloc[0]
        candle_10am_close = candle_10am_data['Close'].iloc[-1]
        candle_10am_high = candle_10am_data['High'].max()
        candle_10am_low = candle_10am_data['Low'].min()
        candle_10am_2pm_direction = 'bullish' if candle_10am_close > candle_10am_open else 'bearish'
        candle_10am_2pm_range_pct = ((candle_10am_high - candle_10am_low) / candle_10am_low) * 100
        
        # Calculate first extreme (high or low formed first)
        # Track when each extreme is hit
        high_time = None
        low_time = None
        
        for idx, row in candle_10am_data.iterrows():
            # Check if new high
            if row['High'] >= candle_10am_high and high_time is None:
                high_time = idx
            # Check if new low
            if row['Low'] <= candle_10am_low and low_time is None:
                low_time = idx
            
            # Stop once both are found
            if high_time is not None and low_time is not None:
                break
        
        # Determine which came first
        if high_time is not None and low_time is not None:
            candle_10am_2pm_first_extreme = 'high' if high_time < low_time else 'low'
        elif high_time is not None:
            candle_10am_2pm_first_extreme = 'high'
        elif low_time is not None:
            candle_10am_2pm_first_extreme = 'low'
        else:
            candle_10am_2pm_first_extreme = None
    else:
        candle_10am_2pm_direction = candle_10am_2pm_range_pct = candle_10am_2pm_first_extreme = None
    
    # ========================================
    # TOUCHES BEFORE 10:30AM (for 10:30+ predictions)
    # ========================================
    before_1030_start = ny_trading_day_start
    before_1030_end = actual_date.replace(hour=10, minute=30, second=0)
    before_1030_data = df.loc[before_1030_start:before_1030_end]
    
    touched_prev_day_high_before_1030 = False
    touched_prev_day_low_before_1030 = False
    touched_prev_week_high_before_1030 = False
    touched_prev_week_low_before_1030 = False
    
    if not before_1030_data.empty and prev_day_high and prev_day_low:
        # Check if prev day high/low was touched
        touched_prev_day_high_before_1030 = check_level_touch(before_1030_data, prev_day_high, threshold_pct=0.05)
        touched_prev_day_low_before_1030 = check_level_touch(before_1030_data, prev_day_low, threshold_pct=0.05)
    
    if not before_1030_data.empty and prev_week_high and prev_week_low:
        # Check if prev week high/low was touched
        touched_prev_week_high_before_1030 = check_level_touch(before_1030_data, prev_week_high, threshold_pct=0.05)
        touched_prev_week_low_before_1030 = check_level_touch(before_1030_data, prev_week_low, threshold_pct=0.05)
    
    # ========================================
    # GAP ANALYSIS
    # ========================================
    if ny_open and prev_day_close:
        gap_pct = ((ny_open - prev_day_close) / prev_day_close) * 100
        
        if gap_pct > 0.1:
            gap_type = 'gap_up'
        elif gap_pct < -0.1:
            gap_type = 'gap_down'
        else:
            gap_type = 'flat'
        
        abs_gap = abs(gap_pct)
        if abs_gap < 0.25:
            gap_size = 'small'
        elif abs_gap < 0.75:
            gap_size = 'medium'
        else:
            gap_size = 'large'
    else:
        gap_pct = gap_type = gap_size = None
    
    # ========================================
    # FLOOR PIVOT POSITION AT NY OPEN
    # ========================================
    if floor_pivots and ny_open:
        ny_open_pivot_position = get_pivot_position(ny_open, floor_pivots)
    else:
        ny_open_pivot_position = None
    
    # ========================================
    # POSITIONING
    # ========================================
    if ny_open and prev_day_high:
        if ny_open > prev_day_high:
            ny_open_vs_prev_day = 'above_high'
        elif ny_open < prev_day_low:
            ny_open_vs_prev_day = 'below_low'
        elif ny_open > prev_day_open:
            ny_open_vs_prev_day = 'above_open'
        elif ny_open < prev_day_open:
            ny_open_vs_prev_day = 'below_open'
        else:
            ny_open_vs_prev_day = 'at_open'
    else:
        ny_open_vs_prev_day = None
    
    if ny_open and prev_week_high:
        if ny_open > prev_week_high:
            ny_open_vs_prev_week = 'above_high'
        elif ny_open < prev_week_low:
            ny_open_vs_prev_week = 'below_low'
        else:
            ny_open_vs_prev_week = 'within'
    else:
        ny_open_vs_prev_week = None
    
    if ny_open and asia_high and london_high:
        overnight_high_combined = max(asia_high, london_high)
        overnight_low_combined = min(asia_low, london_low)
        
        if ny_open > overnight_high_combined:
            ny_open_vs_overnight = 'above'
        elif ny_open < overnight_low_combined:
            ny_open_vs_overnight = 'below'
        else:
            ny_open_vs_overnight = 'within'
    else:
        ny_open_vs_overnight = None
    
    # ========================================
    # DAY OF WEEK
    # ========================================
    day_of_week = actual_date.strftime('%A')
    
    # ========================================
    # VALIDATION
    # ========================================
    required_fields = [
        session_pattern, asia_direction, london_direction, 
        ny_direction, ny_open_vs_overnight
    ]
    
    if all(x is not None for x in required_fields):
        record = {
            'date': actual_date.date(),
            'day_of_week': day_of_week,
            
            # Previous context
            'prev_day_direction': prev_day_direction,
            'prev_day_close': prev_day_close,
            
            # Gap analysis
            'gap_type': gap_type,
            'gap_size': gap_size,
            'gap_pct': gap_pct,
            
            # Positioning
            'ny_open_vs_prev_day': ny_open_vs_prev_day,
            'ny_open_vs_prev_week': ny_open_vs_prev_week,
            'ny_open_vs_overnight': ny_open_vs_overnight,
            
            # Floor pivots
            'ny_open_pivot_position': ny_open_pivot_position,
            'overnight_highest_pivot': overnight_highest_pivot,
            'overnight_lowest_pivot': overnight_lowest_pivot,
            'overnight_touched_pp': overnight_touched_pp,
            'overnight_touched_gz': overnight_touched_gz,
            'overnight_touched_r1': overnight_touched_r1,
            'overnight_touched_r2': overnight_touched_r2,
            'overnight_touched_r3': overnight_touched_r3,
            'overnight_touched_s1': overnight_touched_s1,
            'overnight_touched_s2': overnight_touched_s2,
            'overnight_touched_s3': overnight_touched_s3,
            
            # Overnight level touches
            'overnight_touched_prev_day_high': overnight_touched_prev_day_high,
            'overnight_touched_prev_day_low': overnight_touched_prev_day_low,
            'overnight_touched_prev_week_high': overnight_touched_prev_week_high,
            'overnight_touched_prev_week_low': overnight_touched_prev_week_low,
            'overnight_direction': overnight_direction,
            
            # Session patterns
            'session_pattern': session_pattern,
            'asia_direction': asia_direction,
            'london_direction': london_direction,
            
            # NY ORB (for post-9:45 predictions)
            'ny_orb_direction': ny_orb_direction,
            'ny_orb_first_break': orb_first_break,
            
            # 9:30-9:35 ORB (5-minute, for post-9:45 predictions)
            'orb_935_direction': orb_935_direction,
            'orb_935_first_break': orb_935_first_break,
            
            # Initial Balance (for post-10:30 predictions)
            'ib_direction': ib_direction,
            'ib_range_pct': ib_range_pct,
            
            # Touches before 10:30am (for post-10:30 predictions)
            'touched_prev_day_high_before_1030': touched_prev_day_high_before_1030,
            'touched_prev_day_low_before_1030': touched_prev_day_low_before_1030,
            'touched_prev_week_high_before_1030': touched_prev_week_high_before_1030,
            'touched_prev_week_low_before_1030': touched_prev_week_low_before_1030,
            
            # Outcomes
            'ny_direction': ny_direction,
            'ny_range_pct': ny_range_pct,
            'candle_10am_2pm_direction': candle_10am_2pm_direction,  # NEW: 10am-2pm candle outcome
            'candle_10am_2pm_range_pct': candle_10am_2pm_range_pct,
            'candle_10am_2pm_first_extreme': candle_10am_2pm_first_extreme,  # NEW: which extreme forms first
        }
        
        # Add hourly ranges
        for hour_name, hour_data in hourly_ranges.items():
            if hour_data:
                record[f'{hour_name}_direction'] = hour_data['direction']
            else:
                record[f'{hour_name}_direction'] = None
        
        # Add 4H candles
        for candle_name, candle_data in four_hour_candles.items():
            if candle_data:
                record[f'{candle_name}_direction'] = candle_data['direction']
            else:
                record[f'{candle_name}_direction'] = None
        
        # Add 6-9am range
        record['range_6_9_direction'] = range_6_9_direction
        
        # Add ORB periods
        for orb_name, orb_data in orb_periods.items():
            if orb_data:
                record[f'{orb_name}_direction'] = orb_data['direction']
            else:
                record[f'{orb_name}_direction'] = None
        
        unified_data.append(record)
    else:
        skipped_count += 1

# Convert to DataFrame
unified_df = pd.DataFrame(unified_data)

print(f"\n{'='*80}")
print("DATASET CREATION COMPLETE")
print("="*80)
print(f"✓ Created dataset with {len(unified_df)} valid trading days")
print(f"  Skipped {skipped_count} days due to missing data")
print(f"\n  Total features captured: {len(unified_df.columns)}")

# Save
output_dir = r"C:\Users\tammm\NQ and ES analysis"
unified_df.to_csv(f'{output_dir}\\ultimate_comprehensive_dataset_10am.csv', index=False)
print(f"\n✓ Saved: ultimate_comprehensive_dataset_10am.csv")

print("\n" + "="*80)
print("FEATURE GROUPS")
print("="*80)

feature_groups = {
    'Floor Pivots': [col for col in unified_df.columns if 'pivot' in col or 'gz' in col.lower()],
    'Overnight Touches': [col for col in unified_df.columns if 'overnight_touched' in col],
    'Gap Analysis': [col for col in unified_df.columns if 'gap' in col],
    'Sessions': [col for col in unified_df.columns if any(x in col for x in ['asia', 'london', 'session'])],
    'Hourly Ranges': [col for col in unified_df.columns if any(x in col for x in ['globex', 'midnight', '7am', '8am']) and 'direction' in col],
    '4H Candles': [col for col in unified_df.columns if 'candle_4h' in col],
    'ORBs': [col for col in unified_df.columns if 'orb' in col],
    'NY Session': [col for col in unified_df.columns if col.startswith('ny_') or col.startswith('ib_')],
    'Positioning': [col for col in unified_df.columns if 'vs_' in col],
}

for group_name, features in feature_groups.items():
    if features:
        print(f"\n{group_name}: {len(features)} features")
        for feat in features:
            unique_vals = unified_df[feat].nunique()
            print(f"  • {feat:<50} ({unique_vals} values)")


ULTIMATE COMPREHENSIVE DATASET BUILDER
Analyzing 3785 trading days from 2013-01-01 to 2025-12-31 00:00:00

Features being captured:
  ✓ Floor Pivots (PP, R1-R3, S1-S3, Golden Zones)
  ✓ Opening position relative to floor pivots
  ✓ Overnight pivot touches (which levels hit before 9:30)
  ✓ Previous day/week high/low touches during overnight
  ✓ All hourly ranges + 4H candles (pre-market only)
  ✓ All session patterns + ORBs
  ✓ Initial Balance (9:30-10:30am)
  ✓ NY ORB (9:30-9:45am) + 9:35 ORB (9:30-9:35am)
  ✓ Level touches before 10:30am (prev day/week high/low)
  ✓ Gap analysis vs prev close
  ✓ 10am-2pm candle direction (outcome for 10am entry predictions)
  ✓ 10am-2pm first extreme (high or low formed first)
  NOTE: 10am-2pm and 2pm-5pm candles excluded as FEATURES (too late for prediction)

  Processing day 500...
  Processing day 1000...


In [ ]:
## Baseline Statistics for 10AM ENTRY → 2PM CANDLE CLOSE
## Shows market bias for 10am-2pm candle (NOT full NY session)
## Includes 6am-10am candle as key predictor

import pandas as pd
import numpy as np

input_dir = r"C:\Users\tammm\NQ and ES analysis"

# Load dataset
df = pd.read_csv(f'{input_dir}\\ultimate_comprehensive_dataset.csv')

print("="*100)
print("BASELINE STATISTICS: 10AM ENTRY → 2PM CANDLE")
print("="*100)
print(f"\nAnalyzing {len(df)} trading days from 2013-2025")
print("Outcome: 10am-2pm candle direction (4-hour swing)")
print("NOT analyzing: Full NY session (9:30-4pm)\n")

# Check if data exists
if 'candle_10am_2pm_direction' not in df.columns:
    print("ERROR: candle_10am_2pm_direction not in dataset!")
    print("Run the UPDATED dataset builder first")
    exit()

# ============================================================
# OVERALL BASELINE
# ============================================================
print("="*100)
print("OVERALL BASELINE - 10AM-2PM CANDLE")
print("="*100)

total_days = len(df[df['candle_10am_2pm_direction'].notna()])
bullish_days = len(df[df['candle_10am_2pm_direction'] == 'bullish'])
bearish_days = len(df[df['candle_10am_2pm_direction'] == 'bearish'])

print(f"\n10am-2pm Candle Direction:")
print(f"  Total days: {total_days}")
print(f"  Bullish: {bullish_days} ({bullish_days/total_days*100:.1f}%)")
print(f"  Bearish: {bearish_days} ({bearish_days/total_days*100:.1f}%)")
print(f"\n  ** BASELINE EXPECTATION: {bullish_days/total_days*100:.1f}% bullish **")

print(f"\nComparison to Full NY Session:")
ny_total = len(df[df['ny_direction'].notna()])
ny_bullish = len(df[df['ny_direction'] == 'bullish'])
print(f"  NY Session (9:30-4pm): {ny_bullish/ny_total*100:.1f}% bullish")
print(f"  10am-2pm Candle: {bullish_days/total_days*100:.1f}% bullish")
print(f"  Difference: {(bullish_days/total_days - ny_bullish/ny_total)*100:+.1f}%")

# ============================================================
# BASELINE BY KEY FEATURES
# ============================================================
print("\n" + "="*100)
print("BASELINE BY KEY FEATURES (What % bullish given each condition?)")
print("="*100)

features_to_check = [
    ('candle_4h_6am_direction', '6am-10am Candle'),  # NEW!
    ('ny_orb_direction', 'NY ORB Direction'),
    ('ny_orb_first_break', 'NY ORB First Break'),
    ('orb_935_direction', '9:35 ORB Direction'),
    ('orb_935_first_break', '9:35 ORB First Break'),
    ('overnight_direction', 'Overnight Direction'),
    ('gap_type', 'Gap Type'),
    ('gap_size', 'Gap Size'),
    ('day_of_week', 'Day of Week'),
    ('session_pattern', 'Session Pattern'),
    ('overnight_touched_r1', 'Touched R1 Overnight'),
    ('overnight_highest_pivot', 'Overnight Highest Pivot'),
    ('asia_direction', 'Asia Direction'),
    ('london_direction', 'London Direction'),
]

for feature, description in features_to_check:
    if feature not in df.columns:
        continue
    
    print(f"\n{description} ({feature}):") 
    print("-" * 100)
    
    feature_df = df[df[feature].notna() & df['candle_10am_2pm_direction'].notna()]
    
    if len(feature_df) == 0:
        print("  No data available")
        continue
    
    grouped = feature_df.groupby(feature)['candle_10am_2pm_direction'].apply(
        lambda x: (x == 'bullish').sum() / len(x) * 100
    ).sort_values(ascending=False)
    
    for value, pct in grouped.items():
        count = len(feature_df[feature_df[feature] == value])
        bullish_count = len(feature_df[(feature_df[feature] == value) & (feature_df['candle_10am_2pm_direction'] == 'bullish')])
        value_str = str(value)
        print(f"  {value_str:30s}: {pct:5.1f}% bullish ({bullish_count:4d}/{count:4d} days)")

# ============================================================
# BASELINE FOR COMBINED CONDITIONS
# ============================================================
print("\n" + "="*100)
print("BASELINE FOR KEY COMBINATIONS")
print("="*100)

# 6am candle + ORB
print("\n1. 6am-10am Candle + NY ORB Direction:")
print("-" * 100)
candle_6am_orb_df = df[df['candle_4h_6am_direction'].notna() & df['ny_orb_direction'].notna() & df['candle_10am_2pm_direction'].notna()]
for candle_6am_dir in ['bullish', 'bearish']:
    for orb_dir in ['bullish', 'bearish']:
        subset = candle_6am_orb_df[(candle_6am_orb_df['candle_4h_6am_direction'] == candle_6am_dir) & 
                                    (candle_6am_orb_df['ny_orb_direction'] == orb_dir)]
        if len(subset) > 0:
            bullish_pct = (subset['candle_10am_2pm_direction'] == 'bullish').sum() / len(subset) * 100
            bullish_count = (subset['candle_10am_2pm_direction'] == 'bullish').sum()
            print(f"  6am {candle_6am_dir:7s} + ORB {orb_dir:7s}: {bullish_pct:5.1f}% bullish ({bullish_count:4d}/{len(subset):4d} days)")

# 6am candle + First break
print("\n2. 6am-10am Candle + ORB First Break:")
print("-" * 100)
candle_6am_break_df = df[df['candle_4h_6am_direction'].notna() & df['ny_orb_first_break'].notna() & df['candle_10am_2pm_direction'].notna()]
for candle_6am_dir in ['bullish', 'bearish']:
    for first_break in ['high', 'low']:
        subset = candle_6am_break_df[(candle_6am_break_df['candle_4h_6am_direction'] == candle_6am_dir) & 
                                      (candle_6am_break_df['ny_orb_first_break'] == first_break)]
        if len(subset) > 0:
            bullish_pct = (subset['candle_10am_2pm_direction'] == 'bullish').sum() / len(subset) * 100
            bullish_count = (subset['candle_10am_2pm_direction'] == 'bullish').sum()
            print(f"  6am {candle_6am_dir:7s} + breaks {first_break:4s}: {bullish_pct:5.1f}% bullish ({bullish_count:4d}/{len(subset):4d} days)")

# ORB + First break (for comparison with NY session)
print("\n3. NY ORB Direction + First Break:")
print("-" * 100)
orb_df = df[df['ny_orb_direction'].notna() & df['ny_orb_first_break'].notna() & df['candle_10am_2pm_direction'].notna()]
for orb_dir in ['bullish', 'bearish']:
    for first_break in ['high', 'low']:
        subset = orb_df[(orb_df['ny_orb_direction'] == orb_dir) & (orb_df['ny_orb_first_break'] == first_break)]
        if len(subset) > 0:
            bullish_pct = (subset['candle_10am_2pm_direction'] == 'bullish').sum() / len(subset) * 100
            bullish_count = (subset['candle_10am_2pm_direction'] == 'bullish').sum()
            print(f"  ORB {orb_dir:7s} + breaks {first_break:4s}: {bullish_pct:5.1f}% bullish ({bullish_count:4d}/{len(subset):4d} days)")

# Overnight R1 + 6am candle
print("\n4. Overnight Touched R1 + 6am-10am Candle:")
print("-" * 100)
r1_6am_df = df[df['overnight_touched_r1'].notna() & df['candle_4h_6am_direction'].notna() & df['candle_10am_2pm_direction'].notna()]
for touched in [True, False]:
    for candle_6am_dir in ['bullish', 'bearish']:
        subset = r1_6am_df[(r1_6am_df['overnight_touched_r1'] == touched) & 
                          (r1_6am_df['candle_4h_6am_direction'] == candle_6am_dir)]
        if len(subset) > 0:
            bullish_pct = (subset['candle_10am_2pm_direction'] == 'bullish').sum() / len(subset) * 100
            bullish_count = (subset['candle_10am_2pm_direction'] == 'bullish').sum()
            touched_str = "YES" if touched else "NO"
            print(f"  Touched R1: {touched_str:3s} + 6am {candle_6am_dir:7s}: {bullish_pct:5.1f}% bullish ({bullish_count:4d}/{len(subset):4d} days)")

# ============================================================
# KEY INSIGHTS
# ============================================================
print("\n" + "="*100)
print("KEY INSIGHTS FOR 10AM-2PM CANDLE TRADING")
print("="*100)

baseline = bullish_days/total_days*100

print(f"\n1. 10am-2pm candle baseline: {baseline:.1f}% bullish")
print(f"   → Any pattern showing <{baseline+5:.0f}% bullish has WEAK edge")
print(f"   → Look for patterns >70% (>{baseline+20:.0f}% above baseline) for strong edge")

print(f"\n2. True edge calculation:")
print(f"   → Edge = Pattern Confidence - Baseline ({baseline:.1f}%)")
print(f"   → Example: 75% pattern = {75-baseline:.1f}% true edge")

print(f"\n3. 6am-10am Candle Importance:")
if 'candle_4h_6am_direction' in df.columns:
    candle_6am_df = df[df['candle_4h_6am_direction'].notna() & df['candle_10am_2pm_direction'].notna()]
    for candle_dir in ['bullish', 'bearish']:
        subset = candle_6am_df[candle_6am_df['candle_4h_6am_direction'] == candle_dir]
        if len(subset) > 0:
            bull_pct = (subset['candle_10am_2pm_direction'] == 'bullish').sum() / len(subset) * 100
            edge = bull_pct - baseline
            print(f"   → When 6am candle {candle_dir}: {bull_pct:.1f}% bullish ({edge:+.1f}% edge)")

print(f"\n4. Comparison to Full NY Session:")
print(f"   → 10am-2pm candle may have different bias than full session")
print(f"   → 10am-2pm captures mid-day momentum, not opening/closing volatility")
print(f"   → Patterns may differ significantly from 9:30-4pm patterns")

print("\n" + "="*100)

In [ ]:
## SMART Pattern Discovery for 10AM ENTRY → 2PM CANDLE CLOSE
## Predicts: 10am-2pm candle direction (NOT full NY session)
## Same analysis structure as NY 9:30-4pm, but different outcome variable
## Added feature: 6am-10am candle direction

import pandas as pd
import numpy as np
from itertools import combinations

# Load dataset
input_dir = r"C:\Users\tammm\NQ and ES analysis"
unified_df = pd.read_csv(f'{input_dir}\\ultimate_comprehensive_dataset.csv')

print("="*100)
print("PATTERN DISCOVERY: 10AM ENTRY → 2PM CANDLE CLOSE")
print("="*100)
print(f"Loaded {len(unified_df)} trading days")
print()
print("OUTCOME: candle_10am_2pm_direction (10am-2pm 4H candle)")
print("NOT predicting: ny_direction (9:30-4pm full session)")
print("ADDED FEATURE: candle_4h_6am_direction (6am-10am candle)")
print()
print("STRATEGY:")
print("  1. Mine all 1-feature and 2-feature patterns")
print("  2. For 3-feature patterns, only keep if:")
print("     - Incremental edge from 3rd feature ≥ 4%")
print("     - Sample size ≥ 150 days")
print("     - Absolute confidence improvement ≥ 5%")
print("  3. Never mine 4+ features (diminishing returns)")
print()

# Check if 10am-2pm candle data exists
if 'candle_10am_2pm_direction' not in unified_df.columns:
    print("ERROR: candle_10am_2pm_direction not in dataset!")
    print("Run the UPDATED dataset builder first to add 10am-2pm candle tracking")
    exit()

# Features available for 10AM entry (all features available by 10am)
FEATURES_10AM = [
    'prev_day_direction', 'gap_type', 'gap_size', 'ny_open_vs_prev_day',
    'ny_open_vs_prev_week', 'day_of_week', 'ny_open_pivot_position',
    'overnight_highest_pivot', 'overnight_lowest_pivot', 'overnight_touched_pp',
    'overnight_touched_gz', 'overnight_touched_r1', 'overnight_touched_r2',
    'overnight_touched_r3', 'overnight_touched_s1', 'overnight_touched_s2',
    'overnight_touched_s3', 'overnight_touched_prev_day_high',
    'overnight_touched_prev_day_low', 'overnight_touched_prev_week_high',
    'overnight_touched_prev_week_low', 'overnight_direction', 'session_pattern',
    'asia_direction', 'london_direction', 'globex_6pm_direction',
    'asia_8pm_direction', 'midnight_direction', 'london_2am_direction',
    'hour_7am_direction', 'ny_futures_8am_direction', 'candle_4h_6pm_direction',
    'candle_4h_10pm_direction', 'candle_4h_2am_direction',
    'candle_4h_6am_direction',  # NEW: 6am-10am candle (completes at 10am)
    'globex_orb_direction', 'asia_orb_direction', 'london_orb_direction',
    'premarket_orb_direction', 'range_6_9_direction', 'ny_open_vs_overnight',
    'ny_orb_direction', 'ny_orb_first_break',  # 9:30-9:45 ORB (available by 9:45, but can use for 10am entry)
    'orb_935_direction', 'orb_935_first_break'  # 9:30-9:35 ORB
]

# Filter to available
features_10am = [f for f in FEATURES_10AM if f in unified_df.columns and unified_df[f].notna().sum() > 50]

print(f"Available features: {len(features_10am)}")
if 'candle_4h_6am_direction' in features_10am:
    print("  ✓ 6am-10am candle included")
else:
    print("  ⚠ WARNING: 6am-10am candle not available in dataset")

# Thresholds
MIN_SUPPORT = 0.008
MIN_CONFIDENCE = 0.58
MIN_SAMPLE = 20
MIN_SAMPLE_3FEAT = 150
MIN_INCREMENTAL_3FEAT = 0.04  # 4% incremental improvement
MIN_ABSOLUTE_IMPROVE_3FEAT = 0.05  # 5% absolute improvement

def mine_smart_patterns(feature_list, outcome_var='candle_10am_2pm_direction'):
    """Smart pattern mining with adaptive complexity"""
    
    print(f"\n{'='*100}")
    print(f"MINING PATTERNS FOR 10AM-2PM CANDLE")
    print(f"{'='*100}")
    print(f"Testing {len(feature_list)} features")
    print(f"Outcome: {outcome_var}")
    
    all_patterns = []
    
    # ========================================
    # PHASE 1: Mine ALL 1-feature patterns
    # ========================================
    print(f"\n  Phase 1: Mining 1-feature patterns...")
    
    for f1 in feature_list:
        valid_mask = unified_df[f1].notna() & unified_df[outcome_var].notna()
        subset = unified_df[valid_mask]
        
        if len(subset) < MIN_SAMPLE:
            continue
        
        grouped = subset.groupby([f1, outcome_var]).size()
        
        for (v1, out), count in grouped.items():
            support = count / len(subset)
            if support < MIN_SUPPORT:
                continue
            
            condition_mask = (subset[f1] == v1)
            condition_count = condition_mask.sum()
            
            if condition_count == 0:
                continue
            
            confidence = count / condition_count
            if confidence < MIN_CONFIDENCE:
                continue
            
            baseline_prob = (subset[outcome_var] == out).sum() / len(subset)
            lift = confidence / baseline_prob if baseline_prob > 0 else 0
            
            all_patterns.append({
                'feature_count': 1,
                'f1': f1, 'v1': v1,
                'outcome': out,
                'support': support,
                'confidence': confidence,
                'lift': lift,
                'count': count,
                'condition_count': condition_count,
                'incremental_edge': None,
                'passed_incremental_test': True
            })
    
    print(f"    Found {len([p for p in all_patterns if p['feature_count']==1])} 1-feature patterns")
    
    # ========================================
    # PHASE 2: Mine ALL 2-feature patterns
    # ========================================
    print(f"  Phase 2: Mining 2-feature patterns...")
    
    for f1, f2 in combinations(feature_list, 2):
        valid_mask = unified_df[f1].notna() & unified_df[f2].notna() & unified_df[outcome_var].notna()
        subset = unified_df[valid_mask]
        
        if len(subset) < MIN_SAMPLE:
            continue
        
        grouped = subset.groupby([f1, f2, outcome_var]).size()
        
        for (v1, v2, out), count in grouped.items():
            support = count / len(subset)
            if support < MIN_SUPPORT:
                continue
            
            condition_mask = (subset[f1] == v1) & (subset[f2] == v2)
            condition_count = condition_mask.sum()
            
            if condition_count == 0:
                continue
            
            confidence = count / condition_count
            if confidence < MIN_CONFIDENCE:
                continue
            
            baseline_prob = (subset[outcome_var] == out).sum() / len(subset)
            lift = confidence / baseline_prob if baseline_prob > 0 else 0
            
            all_patterns.append({
                'feature_count': 2,
                'f1': f1, 'v1': v1,
                'f2': f2, 'v2': v2,
                'outcome': out,
                'support': support,
                'confidence': confidence,
                'lift': lift,
                'count': count,
                'condition_count': condition_count,
                'incremental_edge': None,
                'passed_incremental_test': True
            })
    
    print(f"    Found {len([p for p in all_patterns if p['feature_count']==2])} 2-feature patterns")
    
    # ========================================
    # PHASE 3: Mine SELECTIVE 3-feature patterns
    # ========================================
    print(f"  Phase 3: Mining 3-feature patterns (with incremental filtering)...")
    
    checked_3feat = 0
    kept_3feat = 0
    rejected_3feat = 0
    
    for f1, f2, f3 in combinations(feature_list, 3):
        valid_mask = (unified_df[f1].notna() & unified_df[f2].notna() & 
                      unified_df[f3].notna() & unified_df[outcome_var].notna())
        subset = unified_df[valid_mask]
        
        if len(subset) < MIN_SAMPLE:
            continue
        
        grouped = subset.groupby([f1, f2, f3, outcome_var]).size()
        
        for (v1, v2, v3, out), count in grouped.items():
            checked_3feat += 1
            
            support = count / len(subset)
            if support < MIN_SUPPORT:
                continue
            
            condition_mask = (subset[f1] == v1) & (subset[f2] == v2) & (subset[f3] == v3)
            condition_count = condition_mask.sum()
            
            if condition_count < MIN_SAMPLE_3FEAT:
                rejected_3feat += 1
                continue
            
            confidence = count / condition_count
            if confidence < MIN_CONFIDENCE:
                continue
            
            # Calculate baseline from 2-feature pattern
            condition_mask_2feat = (subset[f1] == v1) & (subset[f2] == v2)
            baseline_2feat_subset = subset[condition_mask_2feat]
            
            if len(baseline_2feat_subset) == 0:
                rejected_3feat += 1
                continue
            
            baseline_2feat_conf = len(baseline_2feat_subset[baseline_2feat_subset[outcome_var] == out]) / len(baseline_2feat_subset)
            incremental_edge = confidence - baseline_2feat_conf
            
            # INCREMENTAL TEST
            passed_incremental = (
                abs(incremental_edge) >= MIN_INCREMENTAL_3FEAT and
                abs(confidence - 0.5) >= (abs(baseline_2feat_conf - 0.5) + MIN_ABSOLUTE_IMPROVE_3FEAT)
            )
            
            if not passed_incremental:
                rejected_3feat += 1
                continue
            
            baseline_prob = (subset[outcome_var] == out).sum() / len(subset)
            lift = confidence / baseline_prob if baseline_prob > 0 else 0
            
            kept_3feat += 1
            all_patterns.append({
                'feature_count': 3,
                'f1': f1, 'v1': v1,
                'f2': f2, 'v2': v2,
                'f3': f3, 'v3': v3,
                'outcome': out,
                'support': support,
                'confidence': confidence,
                'lift': lift,
                'count': count,
                'condition_count': condition_count,
                'incremental_edge': incremental_edge,
                'passed_incremental_test': True
            })
    
    print(f"    Checked {checked_3feat} 3-feature combinations")
    print(f"    Kept {kept_3feat} patterns (passed incremental test)")
    print(f"    Rejected {rejected_3feat} patterns (failed incremental test)")
    
    print(f"\n  Total patterns found: {len(all_patterns)}")
    
    return all_patterns

# Mine patterns
print("\n" + "="*100)
print("STARTING PATTERN MINING")
print("="*100)

rules_10am = mine_smart_patterns(features_10am, outcome_var='candle_10am_2pm_direction')

# Convert to dataframe
df_10am = pd.DataFrame(rules_10am) if rules_10am else pd.DataFrame()

# Calculate actionability score
def calc_score(row):
    edge = abs(row['confidence'] - 0.5) * 100
    confidence_score = min(edge * 2, 100)
    sample = row['condition_count']
    if sample < 50:
        freq_score = (sample / 50) * 50
    else:
        freq_score = min(50 + (np.log2(sample / 50) * 25), 100)
    return round((confidence_score * 0.6) + (freq_score * 0.4), 1)

if not df_10am.empty:
    df_10am['score'] = df_10am.apply(calc_score, axis=1)
    df_10am['edge_pct'] = abs(df_10am['confidence'] - 0.5) * 100
    df_10am = df_10am.sort_values('score', ascending=False)

# Save results
print("\n" + "="*100)
print("SAVING RESULTS")
print("="*100)

if not df_10am.empty:
    df_10am.to_csv(f'{input_dir}\\pattern_rules_10am_entry_for_2pm_close.csv', index=False)
    print(f"✓ Saved: pattern_rules_10am_entry_for_2pm_close.csv ({len(df_10am)} patterns)")

# Summary by feature count
print("\n" + "="*100)
print("PATTERN DISTRIBUTION BY COMPLEXITY")
print("="*100)

if not df_10am.empty:
    print(f"\n10AM-2PM CANDLE PATTERNS:")
    print("-"*100)
    
    for feat_count in [1, 2, 3]:
        patterns = df_10am[df_10am['feature_count'] == feat_count]
        
        if len(patterns) == 0:
            continue
        
        bull = patterns[patterns['outcome'] == 'bullish']
        bear = patterns[patterns['outcome'] == 'bearish']
        
        print(f"\n  {feat_count}-Feature Patterns:")
        print(f"    Total: {len(patterns):,} patterns")
        print(f"    Bullish: {len(bull):,} | Avg conf: {bull['confidence'].mean()*100:.1f}% | "
              f"Avg sample: {bull['condition_count'].mean():.0f} days" if len(bull) > 0 else "    Bullish: 0")
        print(f"    Bearish: {len(bear):,} | Avg conf: {bear['confidence'].mean()*100:.1f}% | "
              f"Avg sample: {bear['condition_count'].mean():.0f} days" if len(bear) > 0 else "    Bearish: 0")
        
        if feat_count == 3 and len(patterns) > 0:
            avg_incr = patterns['incremental_edge'].mean()
            print(f"    Avg incremental edge from 3rd feature: {avg_incr*100:+.1f}%")

# Key insights
print("\n\n" + "="*100)
print("KEY INSIGHTS")
print("="*100)

print("\n1. OUTCOME VARIABLE:")
print("   - Predicting: 10am-2pm candle direction (NOT full NY session)")
print("   - This is a 4-hour swing trade target")
print("   - Different from 9:30-4pm prediction")

print("\n2. NEW FEATURE INCLUDED:")
print("   - 6am-10am candle direction (candle_4h_6am_direction)")
print("   - This candle completes exactly at 10am entry time")
print("   - Shows pre-market momentum heading into cash session")

print("\n3. PATTERN QUALITY:")
print("   - 1-feature: Simple, high frequency, moderate edge")
print("   - 2-feature: Best balance of edge and frequency")
print("   - 3-feature: ONLY kept when 3rd feature adds real value (4%+ incremental)")

print("\n4. USAGE:")
print("   - Enter at 10:00am based on pattern conditions")
print("   - Target: 2:00pm close (4-hour swing)")
print("   - Alternative: Use for intraday scalping 10am-2pm")

print("\n" + "="*100)
print("ANALYSIS COMPLETE")
print("="*100)